In [9]:
import pandas as pd
import numpy as np
from matplotlib import pyplot
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import torch
from torch.utils.data import Dataset, DataLoader


In [19]:
# Cargar el dataset
df = pd.read_csv('./datasets/ames-housing-dataset/AmesHousing.csv')

# dimensiones
print("Dimensiones:", df.shape)

# nombres de columnas
print("\nColumnas:")
print(df.columns.tolist())

# vista general
print("\nPrimeras 10 filas:")
print(df.head(10))

#Tipos de datos
n_num = df.select_dtypes(include=[np.number]).shape[1]
n_obj = df.select_dtypes(include=['object']).shape[1]

print(f"\nColumnas numéricas: {n_num}")
print(f"Columnas de tipo objeto: {n_obj}")


Dimensiones: (2930, 82)

Columnas:
['Order', 'PID', 'MS SubClass', 'MS Zoning', 'Lot Frontage', 'Lot Area', 'Street', 'Alley', 'Lot Shape', 'Land Contour', 'Utilities', 'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1', 'Condition 2', 'Bldg Type', 'House Style', 'Overall Qual', 'Overall Cond', 'Year Built', 'Year Remod/Add', 'Roof Style', 'Roof Matl', 'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type', 'Mas Vnr Area', 'Exter Qual', 'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin SF 1', 'BsmtFin Type 2', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF', 'Heating', 'Heating QC', 'Central Air', 'Electrical', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF', 'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath', 'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'Kitchen Qual', 'TotRms AbvGrd', 'Functional', 'Fireplaces', 'Fireplace Qu', 'Garage Type', 'Garage Yr Blt', 'Garage Finish', 'Garage Cars', 'Garage Area', 'Garage Qual', 'Garag

In [20]:
# variable objetivo
target = "SalePrice"

# separar
X = df.drop(columns=[target])
Y = df[target]

# convertir a numpy
X_np = np.array(X)
Y_np = np.array(Y)

print("\nShape X:", X_np.shape)
print("Shape Y:", Y_np.shape)


Shape X: (2930, 81)
Shape Y: (2930,)


In [22]:
# separar columnas
num_cols = X.select_dtypes(include=[np.number]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

# copiar X para trabajar limpio
X_clean = X.copy()

# numéricas → mediana
for col in num_cols:
    X_clean[col] = X_clean[col].fillna(X_clean[col].median())

# categóricas → "None"
for col in cat_cols:
    X_clean[col] = X_clean[col].fillna("None")

# one-hot encoding
X_clean = pd.get_dummies(X_clean, columns=cat_cols, drop_first=False)

print("\nShape después de encoding:", X_clean.shape)


Shape después de encoding: (2930, 321)


In [26]:


# convertir a numpy (por claridad)
X_np = X_clean.values
Y_np = Y.values

X_np = X_clean.values.astype(np.float64)
Y_np = Y.values.astype(np.float64)

# 🔹 SPLIT (antes de normalizar)
X_train, X_test, Y_train, Y_test = train_test_split(
    X_np, Y_np, test_size=0.2, random_state=42
)

print("\nShapes:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Y_train:", Y_train.shape)
print("Y_test:", Y_test.shape)




Shapes:
X_train: (2344, 321)
X_test: (586, 321)
Y_train: (2344,)
Y_test: (586,)


In [ ]:
def featureNormalize(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)

    # evitar división por 0
    sigma[sigma == 0] = 1

    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma


# 🔹 NORMALIZAR USANDO SOLO TRAIN
X_train_norm, mu, sigma = featureNormalize(X_train)

# aplicar mismos parámetros a test
X_test_norm = (X_test - mu) / sigma


# NORMALIZAR Y
y_mu = np.mean(Y_train)
y_sigma = np.std(Y_train)

if y_sigma == 0:
    y_sigma = 1

Y_train_norm = (Y_train - y_mu) / y_sigma
Y_test_norm  = (Y_test - y_mu) / y_sigma


print("\nnormalización:")
print(f"Media X_train_norm: {X_train_norm.mean():.6f}")
print(f"Std X_train_norm:   {X_train_norm.std():.6f}")


Verificación normalización:
Media X_train_norm: -0.000000
Std X_train_norm:   0.993750
